# 02. PyRPMD 化学反应速率计算模块案例 (Bennett–Chandler RPMD Rate & Instanton)

本教程演示 **PyRPMD** 的气相双分子/单分子反应速率计算模块：

$$k_{\mathrm{RPMD}}(T) = \kappa(t_p)\, k_{\mathrm{QTST}}(T)$$

包含四个典型环节：
1. **过渡态验证与反应通道准备 (`pyrpmd.runner.prepare`)**：使用显式反应定义（`schema_version = 2`），验证过渡态一阶鞍点性质（单一负质量加权 Hessian 特征值）、下坡反应通道连通性，以及生成伞形窗口与回穿预算方案。
2. **伞形抽样自由能面（PMF）与 UI / MBAR 估计器 (`[rate.pmf]`)**：沿无穷远渐近归一化反应坐标 $\xi(\mathbf{q})$ 计算质心平均力势 $W(\xi)$，对比伞形积分（Umbrella Integration, `ui`）与多态 Bennett 接受率（`mbar`）估计器，获得量子过渡态理论速率 $k_{\mathrm{QTST}}(T)$。
3. **分割面约束母链与回穿透射系数 (`[recrossing]`)**：在分割面 $\xi^\ddagger$ 上运行 RATTLE 约束热化母链，发射成对的 $\pm\mathbf{p}$ 无约束子轨迹，统计随时间演化的透射系数 $\kappa(t)$ 与速率估计值。
4. **全自动速率工作流配置 (`rate.auto = true`) 与低温环聚合物瞬子（Instanton）隧穿速率 (`pyrpmd.instanton`)**：展示 `automatic.inp` 的自适应参数解析，并在对称 Eckart 势垒上演示交越温度 $T_c$ 以下的半经典环聚合物瞬子速率计算。

In [ ]:
from pathlib import Path
import tempfile
import copy
import json
import numpy as np

import pyrpmd
from pyrpmd import JobSpec
from pyrpmd.runner import prepare, run
from pyrpmd.instanton import crossover_temperature, instanton_seed, instanton_rate_1d
from pyrpmd.potentials.base import PESResult, Potential
from pyrpmd.constants import KB_AU

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    ROOT = Path(pyrpmd.__file__).resolve().parents[2]

print(f"PyRPMD version: {pyrpmd.__version__}")

## 1. 过渡态（TS）检验与反应通道准备 (`prepare`)

`examples/rate/research.inp` 使用 `schema_version = 2` 显式声明反应物片段 `fragments = [[0], [1, 2]]`、产物片段、成键 `forming_bonds = [[0, 1]]` 与断键 `breaking_bonds = [[1, 2]]`。调用 `prepare(..., numerical=True)` 会在启动昂贵的采样前完成：
- 过渡态受力范数与质量加权 Hessian 负特征值个数检查（`saddle_valid`）；
- 沿不稳定简正模式正负方向下坡追踪至渐近区，验证通道与声明的成断键一致（`channel_consistent`）；
- 生成完整的伞形采样窗口计划与回穿预算。

In [ ]:
prep_report = prepare(ROOT / "examples/rate/research.inp", numerical=True)
rxn_prep = prep_report["reaction_preparation"]

print(json.dumps({
    "ready": prep_report["ready"],
    "saddle_valid": rxn_prep["saddle_valid"],
    "channel_consistent": rxn_prep["channel_consistent"],
    "diagnostics": rxn_prep["diagnostics"],
    "umbrella_windows": prep_report["umbrella_preparation"]["windows"],
    "pmf_sampler": prep_report["pmf_plan"]["sampler"],
    "pmf_estimator": prep_report["pmf_plan"]["estimator"],
}, indent=2))

assert prep_report["ready"] is True
assert rxn_prep["saddle_valid"] is True
assert rxn_prep["channel_consistent"] is True

## 2. 完整 RPMD 速率流程：PMF（UI vs MBAR）与回穿因子 $\kappa(t)$

下面我们在临时目录中基于 `examples/rate/pilot.inp` 运行一个紧凑的端到端 RPMD 速率算例（包含伞形采样、分割面约束母链和成对子轨迹回穿），并对比 **UI (Umbrella Integration)** 与 **MBAR** 两种 PMF 估计器。

> **说明**：为保证 Notebook 在数秒内完成，此处采用教学势和精简的窗口/步数；真实体系需按 `docs/convergence.md` 对珠子数 $P$、时间步长、渐近距离 $R_\infty$、窗口重叠及 $\kappa(t)$ 平台期进行系统收敛。

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)

    # 1. 构造紧凑但窗口重叠良好的 UI 速率算例
    spec_ui = JobSpec.from_file(ROOT / "examples/rate/pilot.inp")
    spec_ui.data["umbrella"].update(start=-0.15, stop=1.15, count=14, kappa="1.5 hartree")
    spec_ui.data["sampling"].update(
        equilibration="1 fs", production="6 fs", interval="0.2 fs", replicas=1
    )
    spec_ui.data["analysis"].update(bins=8, bootstrap=4)
    spec_ui.data["recrossing"].update(
        parent_replicas=2,
        parent_samples=2,
        equilibration="1 fs",
        parent_interval="0.5 fs",
        child_pairs=1,
        child_time="2 fs",
        bootstrap=4,
    )

    # 2. 基于 pmf_umbrella_mbar.inp 构造显式 MBAR + UI 对比算例
    spec_mbar = JobSpec.from_file(ROOT / "examples/rate/pmf_umbrella_mbar.inp")
    spec_mbar.data["rate"]["pmf"]["umbrella"].update(
        start=-0.15, stop=1.15, count=14, kappa="1.5 hartree"
    )
    spec_mbar.data["rate"]["pmf"]["sampling"].update(
        equilibration="1 fs", production="6 fs", interval="0.2 fs", replicas=1
    )
    spec_mbar.data["rate"]["pmf"]["analysis"].update(bins=8, bootstrap=4)
    spec_mbar.data["recrossing"].update(
        parent_replicas=2,
        parent_samples=2,
        equilibration="1 fs",
        parent_interval="0.5 fs",
        child_pairs=1,
        child_time="2 fs",
        bootstrap=4,
    )

    run(spec_ui, output=tmp_path / "rate_ui")
    run(spec_mbar, output=tmp_path / "rate_mbar")

    summary_ui = json.loads((tmp_path / "rate_ui/results/summary.json").read_text())
    pmf_ui = json.loads((tmp_path / "rate_ui/rate_pmf.json").read_text())
    summary_mbar = json.loads((tmp_path / "rate_mbar/results/summary.json").read_text())
    pmf_mbar = json.loads((tmp_path / "rate_mbar/rate_pmf.json").read_text())

print(json.dumps({
    "ui_estimator": {
        "pmf_status": pmf_ui["status"],
        "xi_star": pmf_ui["xi_star"],
        "free_energy_barrier_hartree": pmf_ui["free_energy_barrier_hartree"],
        "k_qtst_cm3_s": pmf_ui["k_qtst_cm3_s"],
        "kappa": summary_ui["kappa"],
        "rate_estimate_cm3_s": summary_ui["rate_estimate_cm3_s"],
    },
    "mbar_estimator": {
        "pmf_status": pmf_mbar["status"],
        "xi_star": pmf_mbar["xi_star"],
        "free_energy_barrier_hartree": pmf_mbar["free_energy_barrier_hartree"],
        "k_qtst_cm3_s": pmf_mbar["k_qtst_cm3_s"],
        "kappa": summary_mbar["kappa"],
        "rate_estimate_cm3_s": summary_mbar["rate_estimate_cm3_s"],
    },
}, indent=2))

assert pmf_ui["status"] == "ready"
assert pmf_mbar["status"] == "ready"
assert summary_ui["rate_estimate_cm3_s"] > 0.0
assert summary_mbar["rate_estimate_cm3_s"] > 0.0

## 3. 全自动速率模式 (`[rate] auto = true`) 与精度预设

PyRPMD 支持声明式的全自动速率配置（`examples/rate/automatic.inp`）：只需提供过渡态几何与势能面，设置 `[rate] auto = true`（以及可选的 `precision = "loose" | "normal" | "tight"`），输入解析器即可自动推导自适应伞形采样（AUS）重叠度目标、最小有效样本数（ESS）、覆盖率门限以及回穿阶段参数。

In [ ]:
auto_spec = JobSpec.from_file(ROOT / "examples/rate/automatic.inp")
auto_cfg, auto_sys = auto_spec.resolve()

print(json.dumps({
    "task": auto_cfg["task"],
    "schema_version": auto_cfg["schema_version"],
    "auto_rate_settings": auto_cfg["rate"]["auto"],
    "adaptive_umbrella_targets": {
        k: auto_cfg.get("adaptive_umbrella", {}).get(k)
        for k in ("target_overlap", "min_effective_samples", "target_coverage")
    },
}, indent=2))

## 4. 低温深隧穿区域：环聚合物瞬子（Ring-Polymer Instanton）速率

当温度低于交越温度 $T_c = \frac{\hbar \omega_b}{2\pi k_\mathrm{B}}$ 时，虚时周期轨道（Instanton）主导量子隧穿过程。下面我们在对称一维 Eckart 势垒 $V(x) = V_0 \operatorname{sech}^2(ax)$ 上计算 $T_c$ 并在 $T = 0.5\,T_c$ 下优化环聚合物瞬子轨道，对比离散珠数（$P = 32, 64$）下的瞬子作用量 $S_{\mathrm{inst}}/\hbar$ 与解析连续极限。

In [ ]:
class EckartBarrier(Potential):
    def __init__(self, height=0.01, inverse_width=0.5):
        self.height = height
        self.inverse_width = inverse_width

    def evaluate(self, q):
        t = np.tanh(self.inverse_width * q[:, 0, 0])
        v = self.height * (1.0 - t**2)
        force = np.zeros_like(q)
        force[:, 0, 0] = 2.0 * self.inverse_width * v * t
        return PESResult(v, force)

eckart = EckartBarrier(height=0.01, inverse_width=0.5)
mass = np.array([1822.0])
active = np.array([[True, False, False]])
ts_origin = np.zeros((1, 3))

# 1. 计算交越温度 T_c 并选择深隧穿温度 T = 0.5 * T_c
t_crossover = crossover_temperature(eckart, ts_origin, mass, active_mask=active)
t_tunnel = 0.5 * t_crossover
beta = 1.0 / (KB_AU * t_tunnel)
partition_1d = np.sqrt(mass[0] / (2.0 * np.pi * beta))

# 解析连续瞬子作用量 S / hbar
c_const = 2.0 * np.pi**2 * mass[0] / eckart.inverse_width**2
analytic_action = 2.0 * np.pi / eckart.inverse_width * np.sqrt(2.0 * mass[0] * eckart.height) - c_const / beta

# 2. 在 P=32 与 P=64 下优化环聚合物瞬子并计算隧穿速率
instanton_results = []
for nbeads in (32, 64):
    seed_orbit = instanton_seed(ts_origin, [[1.0, 0.0, 0.0]], nbeads, 2.5)
    res = instanton_rate_1d(
        eckart,
        mass,
        t_tunnel,
        seed_orbit,
        active_mask=active,
        reactant_partition=partition_1d,
        normalization="per_bohr",
        transition_state=ts_origin,
    )
    instanton_results.append({
        "beads": nbeads,
        "action_hbar": res["instanton"]["action_hbar"],
        "action_error": abs(res["instanton"]["action_hbar"] - analytic_action),
        "rate_m_per_s": res["rate"],
        "numerical_converged": res["numerical_converged"],
    })

print(json.dumps({
    "crossover_temperature_K": float(t_crossover),
    "tunnel_temperature_K": float(t_tunnel),
    "analytic_action_hbar": float(analytic_action),
    "instanton_convergence": instanton_results,
}, indent=2))

assert all(item["numerical_converged"] for item in instanton_results)
assert instanton_results[1]["action_error"] < instanton_results[0]["action_error"]